# Transaction Foundation Model on Ray — Part 5: Batch embedding extraction

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min

---

Part 4 trained the foundation model. The recurring production job is to run it over every card and store a fresh embedding for downstream models to consume. That's a heterogeneous job — CPU to read the tokens, GPU for the forward pass — and it streams through a single Ray Data pipeline.

In [ ]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import numpy as np
import pandas as pd

import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT})

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

SCALE = "full"                       # same knob as the earlier parts; mini = tiny, CPU-only
cfg = load_scale(SCALE)
paths = artifact_paths(get_demo_base_dir(), SCALE)

## Turn the trained model into one vector per transaction

The foundation model pretrained on long transaction histories, but the fraud model
wants a fixed vector per transaction. Following NVIDIA's blueprint we embed each
transaction *on its own* — feed `<bos>` + its field tokens + `<eos>` and read the last
token's hidden state. Two choices happen in this step, and both matter: we **sample a
balanced training set first** (every fraud + a ~10% mix of normals) so we embed ~1M
windows plus the held-out test set instead of all ~24M, and we **truncate each window
to its own transaction** (`max_ctx`) so the vector reflects that transaction rather than
a 300-deep running average — the single-transaction embedding is what makes the FM add
signal on top of the raw features downstream.


## Embed every window with a pool of model replicas

This is the stage where Ray Data is genuinely differentiated, not just convenient. The forward pass wants a GPU, but the model is expensive to load — so loading it per batch would be hopeless. The fix is a **stateful** `map_batches`: `EmbeddingExtractor` is a callable class that loads the model **once per replica** in `__init__` and embeds each batch in `__call__`.

`ActorPoolStrategy(min_size=1, max_size=num_workers)` runs a pool of those replicas, growing from one up to `num_workers` as work demands; `num_gpus=1` places each on a GPU (and `num_cpus=1` keeps it on CPU at `mini`). The CPU Parquet read and the GPU forward pass run as one streamed pipeline — batches flow from the readers to the actors through the object store with no intermediate disk writes. Going from CPU here to a pool of GPU replicas at `full` is the `embed` config, not a code change; `scripts/04_extract_embeddings.py` runs this same `EmbeddingExtractor`.

In [ ]:
from src.embed import balanced_eval_sample, extract_embeddings, embedding_health

# NVIDIA NB04: balanced-sample the training set + keep the full holdout BEFORE
# embedding, so we embed ~1M + the held-out test set rather than all ~24M windows,
# and embed each transaction ALONE (max_ctx = BOS + its ~12 field tokens + EOS).
# The single-transaction embedding is what the fraud model consumes downstream.
eb = cfg["embed"]
if not os.path.exists(paths["embeddings"]):
    sampled = balanced_eval_sample(paths["tokenized_eval"], balanced_train=eb["balanced_train"])
    extract_embeddings(
        ds=sampled,
        checkpoint_dir=paths["checkpoint"],
        output_path=paths["embeddings"],
        num_workers=eb["num_workers"],
        use_gpu=eb["use_gpu"],
        batch_size=eb["batch_size"],
        max_ctx=eb["max_ctx"],
    )
else:
    print(f"embeddings exist at {paths['embeddings']} — delete to re-embed")


## Did the embeddings actually learn anything?

The classic self-supervised failure is silent **representation collapse**: every customer maps to nearly the same vector while the loss looks fine. A cheap guard is to sample the embeddings and check two numbers — mean pairwise cosine similarity (→1.0 means collapse) and mean feature variance (→0 means collapse). `embedding_health` reports both.

In [ ]:
embedding_health(paths["embeddings"])

# Don't pull the whole table into pandas just to peek at it — at `full` that's
# millions of rows, each carrying a tensor column that pandas materializes as a
# separate numpy object. Read the count from Parquet metadata and grab one row.
eds = ray.data.read_parquet(paths["embeddings"])
n_windows = eds.count()                            # metadata only, no vectors read
row = eds.limit(1).to_pandas().iloc[0]             # a single window for the example
vec0 = np.asarray(row["embedding"])                # Ray stores this as a tensor column
carried = [c for c in row.index if c != "embedding"]

print(f"\n{n_windows:,} window embeddings · dim {vec0.shape[-1]}")
print(f"carried alongside each vector (for Part 6): {carried}")

print(f"\nexample — card {int(row['card_id'])}, label {int(row['label'])}, split {row['split']}")
print(f"  embedding[:8] = {vec0[:8].round(3).tolist()}")

## Takeaways

**Ray Data**
- A stateful `map_batches` (a callable class + `ActorPoolStrategy`) loads the model **once per replica**, not once per batch — the difference between usable and hopeless for GPU inference.
- `num_gpus` on `map_batches` places each replica on a GPU; the CPU read and GPU forward pass run as one streamed pipeline with no intermediate disk. This heterogeneous CPU→GPU shape is where Ray Data genuinely earns its keep.
- The same code goes from one CPU replica at `mini` to a pool of GPU replicas at `full` by changing the `embed` config; the notebook runs the same `EmbeddingExtractor` as `scripts/04_extract_embeddings.py`.

**The embeddings**
- One pooled vector per window (`"last"` pooling = the most recent transaction's hidden state, aligned with the per-transaction fraud label).
- Each vector is written next to its `label`, `split`, `weight`, and raw target features, so the downstream stage needs no re-reads.
- A quick collapse check (mean pairwise cosine, feature variance) catches the silent self-supervised failure mode before it reaches the downstream model.

---

## Next

**Part 6 — Downstream fraud: raw vs FM vs fusion**: train the same XGBoost recipe on three feature sets — raw transaction fields, the FM embedding, and the two fused — and measure the lift at natural fraud prevalence with PR-AUC.